In [1]:
# docker file used for this notebook for pytorch 2.0
# docker pull cnstark/pytorch:2.0.1-py3.10.11-cuda11.8.0-ubuntu22.04

In [2]:
import torch
print(torch.__version__)

2.3.0a0+6ddf5cf85e.nv24.04



# Семинар torch.compile 

Cеминар основан на документации [1] и ноутбуке [2].<br>

[1] https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html <br>
[2] https://colab.research.google.com/github/PyTorchKorea/tutorials-kr/blob/master/docs/_downloads/96ad88eb476f41a5403dcdade086afb8/torch_compile_tutorial.ipynb <br>

``torch.compile`` компилятор для ускорения кода на PyTorch. <br>
Это JIT-компилятор с использованием оптимизированных кернелей,
который требует минимального изменения кода. <br>

В этом семинаре мы рассмотрим базовое использование ``torch.compile``, а так же сравним его с [TorchScript](https://pytorch.org/docs/stable/jit.html) и
[FX Tracing](https://pytorch.org/docs/stable/fx.html#torch.fx.symbolic_trace).

**Содержание**

- Базовый пример
- Ускорение с помощью ``torch.compile``
- Сравнение с TorchScript и FX Tracing
- Заключение

**Required pip Dependencies**

- ``torch >= 2.0``
- ``torchvision``
- ``numpy``
- ``scipy``
- ``tabulate``

docker file used for this notebook for pytorch 2.0: <br>
``docker pull cnstark/pytorch:2.0.1-py3.10.11-cuda11.8.0-ubuntu22.04``





ПРИМЕЧАНИЕ. Результаты зависят от версии GPU, что бы воспроизвести результаты команда ``torch.compile`` рекомендует использование NVIDIA (H100, A100 или V100). 

## Базовый пример

Произвольные функции Python можно оптимизировать, передав вызываемый объект в
``torch.compile``. Затем мы можем вызвать возвращенную оптимизированную
функцию вместо исходной.



In [3]:
import torch

def foo(x, y):
    a = torch.sin(x)
    b = torch.cos(y)
    return a + b

opt_foo = torch.compile(foo)

print(opt_foo(torch.randn(10, 10), torch.randn(10, 10)))

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[ 1.4479e-02, -4.0956e-01,  8.9599e-01,  8.3348e-01,  1.9756e+00,
          9.1979e-01,  1.0492e+00,  3.3680e-01,  9.4792e-01, -7.0554e-01],
        [-5.7015e-01,  9.1068e-01,  5.3560e-01,  1.1650e+00,  4.6061e-01,
          1.6078e+00,  9.7864e-02,  7.6591e-02, -1.6105e-01,  6.2816e-01],
        [ 8.5078e-01, -1.3855e-03, -6.1361e-01,  6.2968e-01,  1.7911e+00,
          1.9602e+00,  4.0344e-01,  1.0502e+00,  1.7362e+00,  6.8823e-01],
        [ 1.3535e+00,  1.0859e+00, -7.0885e-01, -8.0579e-02, -1.4691e+00,
          8.2291e-01,  7.5273e-01,  1.7696e+00,  1.1447e+00,  1.8427e+00],
        [-1.8422e-01, -3.6353e-02,  1.5535e+00,  1.3854e-01,  1.7834e+00,
          1.0076e+00,  9.7050e-02, -7.1476e-03,  1.2463e+00,  1.0727e+00],
        [-1.1644e-01,  6.4685e-01, -6.7487e-02, -4.8843e-02,  2.8316e-01,
          4.4767e-02,  1.3634e-01, -9.6914e-01,  1.8826e+00,  1.0600e+00],
        [-8.2864e-01,  1.8747e+00, -2.7044e-01,  4.5695e-01,  1.7048e+00,
          3.2712e-02,  1.3635e+0

Так же можно использовать декоратор

In [4]:
@torch.compile
def opt_foo2(x, y):
    a = torch.sin(x)
    b = torch.cos(y)
    return a + b
print(opt_foo2(torch.randn(10, 10), torch.randn(10, 10)))

tensor([[ 4.3356e-01, -4.9096e-01, -1.4848e+00,  3.9684e-02,  9.8495e-01,
          1.0708e+00,  1.9664e+00, -2.0685e-02,  4.3578e-01,  1.8935e+00],
        [ 1.2102e+00,  6.5481e-01,  1.5728e+00,  1.2823e+00,  2.8214e-03,
         -2.7513e-01, -3.5863e-01,  9.6984e-01,  1.3246e-01,  6.9340e-01],
        [-1.6163e-01,  6.2003e-01,  4.9083e-01,  1.4977e-02,  1.4272e-01,
          4.6773e-01, -7.7897e-01, -3.6293e-04,  1.6590e+00, -2.1414e-01],
        [ 8.2279e-01,  8.0079e-01,  5.6301e-01, -1.5357e-01, -6.0512e-02,
          1.8688e+00, -1.0327e+00,  1.1402e+00,  2.5474e-03,  5.6185e-01],
        [ 6.7506e-01,  1.8507e+00,  9.8079e-02, -1.0871e-01,  1.9568e-01,
         -1.6611e-01,  1.5702e-01,  1.2110e+00,  1.3004e+00, -3.8495e-02],
        [ 8.0977e-02, -5.7162e-02, -5.5322e-02, -5.8731e-01,  3.7260e-01,
          2.6502e-01, -4.9054e-01,  1.2001e+00,  6.5512e-01,  1.6823e+00],
        [-9.7112e-01, -3.6403e-01, -1.4094e+00, -2.3573e-02,  1.7007e+00,
         -3.0796e-01, -8.3894e-0

Мы можем оптимизировать целый модуль ``torch.nn.Module``.



In [5]:
class MyModule(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = torch.nn.Linear(100, 10)

    def forward(self, x):
        return torch.nn.functional.relu(self.lin(x))

mod = MyModule()

opt_mod = torch.compile(mod)
print(opt_mod(torch.randn(10, 100)))

tensor([[0.0000, 0.0000, 0.8847, 0.4645, 0.0000, 0.0000, 0.5090, 0.2111, 0.3361,
         0.3766],
        [0.0000, 0.0655, 0.0000, 0.0000, 0.0000, 0.0000, 1.0180, 0.0000, 1.0811,
         0.0000],
        [0.1067, 0.1260, 0.5498, 0.0000, 0.0000, 1.3791, 1.0794, 0.0000, 0.0000,
         0.0000],
        [0.3888, 0.1854, 0.1622, 0.0000, 0.0000, 0.5605, 0.0000, 0.3148, 0.0000,
         0.0696],
        [0.0000, 0.0506, 0.0000, 0.3052, 0.0000, 0.1279, 0.0000, 0.3276, 0.0521,
         0.1324],
        [0.0000, 0.0000, 0.0060, 0.7962, 0.0000, 0.2488, 0.5349, 0.0000, 0.1528,
         0.0000],
        [0.0000, 0.5542, 0.0000, 0.1410, 0.0000, 0.0000, 0.6408, 0.9086, 0.0000,
         0.0866],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.8545, 0.0741, 0.8252, 0.1845, 0.8393,
         0.0000],
        [0.0000, 0.0000, 0.1853, 0.0000, 0.6124, 0.0000, 0.0000, 0.6427, 0.0000,
         0.0000],
        [0.0000, 0.7550, 0.0000, 1.0601, 0.0000, 0.0000, 0.4304, 0.0000, 0.0000,
         0.0000]], grad_fn=<


## Ускорение для инференса

Давайте теперь посмотрим, как использование ``torch.compile`` может ускорить <br>
модели. Мы сравним стандартный eager режим и ``torch.compile`` для инфиренса и для обучения DenseNet на случайных данных.

Но прежде чем мы начнем, нам нужно определить некоторые служебные функции.


In [6]:
# Returns the result of running `fn()` and the time it took for `fn()` to run,
# in seconds. We use CUDA events and synchronization for the most accurate
# measurements.
def timed(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    result = fn()
    end.record()
    torch.cuda.synchronize()
    return result, start.elapsed_time(end) / 1000

# Generates random input and targets data for the model, where `b` is
# batch size.
def generate_data(b):
    return (
        torch.randn(b, 3, 128, 128).to(torch.float32).to('cuda:1'),
        torch.randint(1000, (b,)).to('cuda:1'),
    )

from torchvision.models import densenet121
def init_model():
    return densenet121().to(torch.float32).to('cuda:1')

Первым делом мы сравним инференс, но прежде посмотрим сколько времени занимает компиляция.

В  ``torch.compile`` мы используем некотрые параметры для переменной ``mode``, мы обсудим их позже.


In [7]:
def evaluate(mod, inp):
    return mod(inp)

model = init_model()

# Reset since we are using a different model.
import torch._dynamo
torch._dynamo.reset()


evaluate_opt = torch.compile(evaluate)
evaluate_opt1 = torch.compile(evaluate, mode="reduce-overhead")
evaluate_opt2 = torch.compile(evaluate, mode="max-autotune")


inp = generate_data(16)[0]

# One Iteration for each function
print("standard:", timed(lambda: evaluate_opt(model, inp))[1])
# print("reduce-overhead:", timed(lambda: evaluate_opt1(model, inp))[1])
print("max-autotune:", timed(lambda: evaluate_opt2(model, inp))[1])
print("original", timed(lambda: evaluate(model, inp))[1])

standard: 82.2276640625


W0320 11:47:58.035000 139894021054912 torch/_inductor/utils.py:936] [0/1] not enough SMs to use max_autotune_gemm mode


max-autotune: 76.517390625
original 0.0992422103881836


In [8]:
# One Iteration for each function 
print("standard:", timed(lambda: evaluate_opt(model, inp))[1])
# print("reduce-overhead:", timed(lambda: evaluate_opt1(model, inp))[1])
print("max-autotune:", timed(lambda: evaluate_opt2(model, inp))[1])
print("original", timed(lambda: evaluate(model, inp))[1])

standard: 0.0420695686340332
max-autotune: 0.8965526733398438
original 0.05130246353149414


Обратите внимание, что выполнение ``torch.compile`` занимает намного больше времени <br>
по сравнению с ``eager mode``. Это потому, что ``torch.compile`` занимается компиляцией и  <br>
оптимизирует ядра по мере ее выполнения.  Здесь мы запускаем нашу модель один раз. <br>

Если мы будем запускать нашу модель несколько раз, то после компиляции, которая происходит на первом этапе мы увидим ускорение. <br>

In [9]:
N_ITERS = 10

eager_times = []
compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)[0]
    _, eager_time = timed(lambda: evaluate(model, inp))
    eager_times.append(eager_time)
    print(f"eager eval time {i}: {eager_time}")

print("~" * 10)

compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)[0]
    _, compile_time = timed(lambda: evaluate_opt(model, inp))
    compile_times.append(compile_time)
    print(f"compile eval time {i}: {compile_time}")
print("~" * 10)

import numpy as np
eager_med = np.median(eager_times)
compile_med = np.median(compile_times)
speedup = eager_med / compile_med
print(f"(eval) eager median: {eager_med}, compile median: {compile_med}, speedup: {speedup}x")
print("~" * 10)

eager eval time 0: 0.036362239837646484
eager eval time 1: 0.04495727920532227
eager eval time 2: 0.03407468795776367
eager eval time 3: 0.032941120147705075
eager eval time 4: 0.03336492919921875
eager eval time 5: 0.04599980926513672
eager eval time 6: 0.032825950622558595
eager eval time 7: 0.03795235061645508
eager eval time 8: 0.041250335693359376
eager eval time 9: 0.04075321578979492
~~~~~~~~~~
compile eval time 0: 0.048710113525390626
compile eval time 1: 0.057737377166748045
compile eval time 2: 0.04584447860717773
compile eval time 3: 0.045329471588134766
compile eval time 4: 0.04701388931274414
compile eval time 5: 0.03933731079101563
compile eval time 6: 0.039276481628417965
compile eval time 7: 0.03675775909423828
compile eval time 8: 0.0375928955078125
compile eval time 9: 0.03916799926757813
~~~~~~~~~~
(eval) eager median: 0.03715729522705078, compile median: 0.0423333911895752, speedup: 0.8777301837373459x
~~~~~~~~~~


In [10]:
N_ITERS = 10

model = init_model()
opt = torch.optim.Adam(model.parameters())

def train(mod, data):
    opt.zero_grad(True)
    pred = mod(data[0])
    loss = torch.nn.CrossEntropyLoss()(pred, data[1])
    loss.backward()
    opt.step()

eager_times = []
for i in range(N_ITERS):
    inp = generate_data(16)
    _, eager_time = timed(lambda: train(model, inp))
    eager_times.append(eager_time)
    print(f"eager train time {i}: {eager_time}")
print("~" * 10)

model = init_model()
opt = torch.optim.Adam(model.parameters())
train_opt = torch.compile(train)

compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)
    _, compile_time = timed(lambda: train_opt(model, inp))
    compile_times.append(compile_time)
    print(f"compile train time {i}: {compile_time}")
print("~" * 10)

eager_med = np.median(eager_times)
compile_med = np.median(compile_times)
speedup = eager_med / compile_med
print(f"(train) eager median: {eager_med}, compile median: {compile_med}, speedup: {speedup}x")
print("~" * 10)

eager train time 0: 0.3775938415527344
eager train time 1: 0.08219635009765625
eager train time 2: 0.07891145324707032
eager train time 3: 0.07858739471435547
eager train time 4: 0.07869993591308594
eager train time 5: 0.07690476989746094
eager train time 6: 0.08000518035888672
eager train time 7: 0.0780670394897461
eager train time 8: 0.0776797103881836
eager train time 9: 0.07977436828613281
~~~~~~~~~~


W0320 11:51:11.989000 139894021054912 torch/_logging/_internal.py:1020] [3/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored


compile train time 0: 256.908875
compile train time 1: 0.11412258911132812
compile train time 2: 0.09625440216064453
compile train time 3: 0.09145753479003907
compile train time 4: 0.09293456268310547
compile train time 5: 0.09084722900390625
compile train time 6: 0.08764415740966797
compile train time 7: 0.08112947082519531
compile train time 8: 0.08835110473632812
compile train time 9: 0.0852093734741211
~~~~~~~~~~
(train) eager median: 0.07880569458007813, compile median: 0.09115238189697267, speedup: 0.8645489337749869x
~~~~~~~~~~


In [11]:
compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)
    _, compile_time = timed(lambda: train_opt(model, inp))
    compile_times.append(compile_time)
    print(f"compile train time {i}: {compile_time}")
print("~" * 10)

eager_med = np.median(eager_times)
compile_med = np.median(compile_times)
speedup = eager_med / compile_med
print(f"(train) eager median: {eager_med}, compile median: {compile_med}, speedup: {speedup}x")
print("~" * 10)

compile train time 0: 0.08957532501220702
compile train time 1: 0.08118672180175782
compile train time 2: 0.07858790588378907
compile train time 3: 0.08101270294189453
compile train time 4: 0.0809697265625
compile train time 5: 0.07914691162109375
compile train time 6: 0.08454252624511718
compile train time 7: 0.08014361572265626
compile train time 8: 0.08014591979980469
compile train time 9: 0.0827394561767578
~~~~~~~~~~
(train) eager median: 0.07880569458007813, compile median: 0.08099121475219725, speedup: 0.9730153427281466x
~~~~~~~~~~


Опять, мы видим, что ``torch.compile`` первая итерация занимает больше времени.
итерации, так как она должна скомпилировать модель, но на последующих итерациях мы видим
значительное ускорение по сравнению с нетерпеливым.


## Cравнение с TorchScript и FX Tracing

Мы видели что ``torch.compile`` может ускорить вычисления. <br>


Прежде всего, преимущество ``torch.compile`` заключается в его способности работать
почти с произвольным кодом на Python с его минимальными изменениями. <br>

А так же ``torch.compile`` может работать с data-dependent control flow ``if x.sum() < 0:``.



In [12]:
def f1(x):
    if x < 10:
        return torch.tensor(0.)
    return x

# Test that `fn1` and `fn2` return the same result, given
# the same arguments `args`. Typically, `fn1` will be an eager function
# while `fn2` will be a compiled function (torch.compile, TorchScript, or FX graph).
def test_fns(fn1, fn2, args):
    out1 = fn1(*args)
    out2 = fn2(*args)
    return print(out1, out2)

inp1 = torch.tensor(5)
inp2 = torch.tensor(15)

TorchScript tracing ``f1`` привдет к неправильным результатам, так  ак зафиксируется детерменированный проход по данным.



In [13]:
traced_f1 = torch.jit.trace(f1, (inp1))
f1(inp1),traced_f1(inp1)

/tmp/ipykernel_703890/2755518421.py:2: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x < 10:
/tmp/ipykernel_703890/2755518421.py:3: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  return torch.tensor(0.)


(tensor(0.), tensor(0.))

In [14]:
f1(inp2),traced_f1(inp2) 

(tensor(15), tensor(0.))

```
TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.sum() < 0:
```

FX tracing ``f1`` приводит к ошибке из-за присутствия поток управления, зависящий от данных.



In [15]:
import traceback as tb
try:
    torch.fx.symbolic_trace(f1)
except:
    tb.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_703890/1451191637.py", line 3, in <module>
    torch.fx.symbolic_trace(f1)
  File "/usr/local/lib/python3.10/dist-packages/torch/fx/_symbolic_trace.py", line 1193, in symbolic_trace
    graph = tracer.trace(root, concrete_args)
  File "/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py", line 437, in _fn
    return fn(*args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/torch/_dynamo/external_utils.py", line 36, in inner
    return fn(*args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/torch/fx/_symbolic_trace.py", line 793, in trace
    (self.create_arg(fn(*args)),),
  File "/tmp/ipykernel_703890/2755518421.py", line 2, in f1
    if x < 10:
  File "/usr/local/lib/python3.10/dist-packages/torch/fx/proxy.py", line 443, in __bool__
    return self.tracer.to_bool(self)
  File "/usr/local/lib/python3.10/dist-packages/torch/fx/proxy.py", line 303, in to_bool
    raise TraceError('symbolically

Если мы предоставим значение для ``x`` при попытке  FX  trace ``f1``, тогда
мы сталкиваемся с той же проблемой, что и при TorchScript, из за зависимости графа от данных.

In [16]:
fx_f1 = torch.fx.symbolic_trace(f1, concrete_args={"x": inp1})
fx_f1(inp1), fx_f1(inp2)

/usr/local/lib/python3.10/dist-packages/torch/fx/_symbolic_trace.py:862: UserWarning: Was not able to add assertion to guarantee correct input x to specialized function. It is up to the user to make sure that your inputs match the inputs you specialized the function with.
  warnings.warn(


(tensor(0.), tensor(0.))

В это время ``torch.compile`` корректно обрабатывает динамический control-flow.



In [17]:
# Reset since we are using a different mode.
torch._dynamo.reset()

compile_f1 = torch.compile(f1)
compile_f1(inp1), compile_f1(inp2)

(tensor(0.), tensor(15))

TorchScript scripting может работать в этой ситуации, но нам нужно будет адаптировать код и убедиться что у нас используется статическое типизирование данных.



In [18]:
script_f1 = torch.jit.script(f1)
script_f1(inp1), script_f1(inp2)

(tensor(0.), tensor(15))

In [22]:
def f2(x, y):
    return x + y

inp1 = torch.randn(5, 5)
inp2 = 3

script_f2 = torch.jit.script(f2)
try:
    script_f2(inp1, inp2)
except:
    tb.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_703890/895162831.py", line 9, in <module>
    script_f2(inp1, inp2)
RuntimeError: f2() Expected a value of type 'Tensor (inferred)' for argument 'y' but instead found type 'int'.
Inferred 'y' to be of type 'Tensor' because it was not annotated with an explicit type.
Position: 1
Value: 3
Declaration: f2(Tensor x, Tensor y) -> Tensor
Cast error details: Unable to cast 3 to Tensor


In [23]:
inp2 = torch.tensor(3.0) 


script_f2 = torch.jit.script(f2)
try:
    script_f2(inp1, inp2)
except:
    tb.print_exc()

Другое преимущество ``torch.compile``  это возможность использлвать больше функций.


In [24]:
import scipy

def f3(x):
    x = x * 2
    x = scipy.fft.dct(x.numpy())
    x = torch.from_numpy(x)
    x = x * 2
    return x

TorchScript обрабатывает результаты вызовов функций, отличных от PyTorch.
как константы, и поэтому результаты могут быть ошибочными.

In [31]:
import numpy as np

In [32]:
inp1 = torch.randn(5, 5)
inp2 = torch.randn(5, 5)
traced_f3 = torch.jit.trace(f3, (inp1,))

/tmp/ipykernel_703890/2558593883.py:5: TracerWarning: Converting a tensor to a NumPy array might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  x = scipy.fft.dct(x.numpy())
/tmp/ipykernel_703890/2558593883.py:6: TracerWarning: torch.from_numpy results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  x = torch.from_numpy(x)


In [33]:
f3(inp2)

tensor([[ -5.8879,  -3.5208,  -5.0445,  -6.7654, -19.7858],
        [  4.9808,  -7.8016, -17.8780, -19.5217,  -8.8669],
        [  2.9493,  12.3907, -14.3234,  -3.1903,  -1.4461],
        [  2.0236,  17.7643,   4.1332,  -0.9385,  11.2552],
        [ -8.3631, -15.4295,   2.3594,   1.8688, -15.2161]])

In [34]:
traced_f3(inp2)

tensor([[-15.2430,   2.3558, -29.8826,  14.5607,  -7.7809],
        [  1.5314,   6.6989,  -9.5644,   9.2357, -16.7371],
        [ -4.8310,  -4.1515,   5.4726,   2.8808,  -2.9561],
        [ 24.7039, -15.7964,   2.2785,   4.4187,  10.5870],
        [-43.9447, -16.8784,  19.2634,  16.8805,   6.7792]])

``torch.compile`` 


In [35]:
inp2 = torch.randn(2, 2)
compile_f3 = torch.compile(f3)
compile_f3(inp1), compile_f3(inp2), np.allclose(f3(inp2), compile_f3(inp2))

(tensor([[-15.2430,   2.3558, -29.8826,  14.5607,  -7.7809],
         [  1.5314,   6.6989,  -9.5644,   9.2357, -16.7371],
         [ -4.8310,  -4.1515,   5.4726,   2.8808,  -2.9561],
         [ 24.7039, -15.7964,   2.2785,   4.4187,  10.5870],
         [-43.9447, -16.8784,  19.2634,  16.8805,   6.7792]]),
 tensor([[-19.4982,   5.9068],
         [ -4.8008,  -3.2399]]),
 True)

## TorchDynamo и FX Graphs

Одна из важных компонент ``torch.compile`` это TorchDynamo.
TorchDynamo отвечает за JIT компиляцию кода на Python в
[FX graphs](https://pytorch.org/docs/stable/fx.html#torch.fx.Graph), который далее может быть оптимизирован. TorchDynamo выделяет FX графы анализруя Питоновский байткод во время выполнения, а так же отслеживает вызовы к функциям PyTorch.

TorchInductor, другая компонента ``torch.compile``,
конвертирует FX граф в оптимизированные кернели, а
TorchDynamo позволяет использовать различные бекнеды.  <br>


Давайте создадим свой бекенд, который выводит FX граф и не оптимизированный forward (что-то типа принта, но вызывывается он TorchDynamo). 

In [36]:
from typing import List

def custom_backend(gm: torch.fx.GraphModule, example_inputs: List[torch.Tensor]):
    print("custom backend called with FX graph:")
    gm.graph.print_tabular()
    return gm.forward

# Reset since we are using a different backend.
torch._dynamo.reset()

opt_model = torch.compile(init_model(), backend=custom_backend)
opt_model(generate_data(16)[0])

custom backend called with FX graph:
opcode         name                                               target                                                      args                                                                                                                                                                                                                                                                                                                                                                                                                                                                kwargs
-------------  -------------------------------------------------  ----------------------------------------------------------  ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

tensor([[-0.0164,  0.2022,  0.3542,  ..., -0.0837, -0.3593,  0.5004],
        [ 0.0868, -0.1411,  0.3950,  ...,  0.1640, -0.2762,  0.4251],
        [-0.1092,  0.0267,  0.4053,  ..., -0.0774, -0.4176,  0.4342],
        ...,
        [ 0.0387,  0.0506,  0.3004,  ..., -0.0169, -0.4291,  0.3396],
        [ 0.0056,  0.0230,  0.1657,  ...,  0.0206, -0.4875,  0.4720],
        [ 0.0840,  0.0479,  0.3622,  ...,  0.0627, -0.3796,  0.5323]],
       device='cuda:1', grad_fn=<AddmmBackward0>)


Используя наш собственный бэкэнд, мы теперь можем увидеть, как TorchDynamo работает с ситуацией, когда есть зависимость от данных. Рассмотрим:
``if b.sum() < 0``. 



In [37]:
def bar(a, b):
    x = a / (torch.abs(a) + 1)
    if b.sum() < 0:
        b = b * -1
    return x * b

opt_bar = torch.compile(bar, backend=custom_backend)
inp1 = torch.randn(10)
inp2 = torch.randn(10)
opt_bar(inp1, inp2)
opt_bar(inp1, -inp2)

custom backend called with FX graph:
opcode         name    target                                                  args         kwargs
-------------  ------  ------------------------------------------------------  -----------  --------
placeholder    l_a_    L_a_                                                    ()           {}
placeholder    l_b_    L_b_                                                    ()           {}
call_function  abs_1   <built-in method abs of type object at 0x7f3b69d8c4a0>  (l_a_,)      {}
call_function  add     <built-in function add>                                 (abs_1, 1)   {}
call_function  x       <built-in function truediv>                             (l_a_, add)  {}
call_method    sum_1   sum                                                     (l_b_,)      {}
call_function  lt      <built-in function lt>                                  (sum_1, 0)   {}
output         output  output                                                  ((x, lt),)   {}
cus

tensor([-0.7088, -0.3590, -0.4616,  0.1233, -0.1748, -0.5861, -0.0424,  1.0782,
        -0.1961,  0.5017])

Мы видим что TorchDynamo разбил наш код на три части, которые соотвествуют:


1. ``x = a / (torch.abs(a) + 1)``
2. ``b = b * -1; return x * b``
3. ``return x * b``

Когда TorchDynamo встречает не поддерживаемые в функции, например зависящие от входных данных, он  разбивает код на части, дает передает все в стандатный Python интерпритатор, а потом возвращается к графу.

Давайте на примере разберемся, как TorchDynamo пройдет через ``<``.
Если ``b.sum() < 0``, то TorchDynamo запустит граф 1, даст
Python определить результат условного выражения, а затем запустит
граф 2. С другой стороны, если ``не b.sum() < 0``, то TorchDynamo
запустил бы граф 1, позволил Python определить результат условного выражения, затем
запустил график 3. <br>


Это подчеркивает основное различие между TorchDynamo и предыдущим  компиляторами для PyTorch.

Предыдущие решения либо упадут с ошибков, либо не правильно скомпилируются и ничего не произойдет.
TorchDynamo, с другой стороны, просто разобьет граф вычислений на части.

Что бы посмотреть как TorchDynamo разбивает граф на части мы можем вызвать: ``torch._dynamo.explain``



In [38]:
# Reset since we are using a different backend.
torch._dynamo.reset()
a  = torch._dynamo.explain(
    bar, torch.randn(10), torch.randn(10)
)

/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:750: UserWarning: explain(f, *args, **kwargs) is deprecated, use explain(f)(*args, **kwargs) instead.  If you don't migrate, we may break your explain call in the future if your user defined kwargs conflict with future kwargs added to explain(f).
  warnings.warn(


Чтобы максимизировать ускорение, разрывы графика должны быть ограничены.
Мы можем заставить TorchDynamo выдавать ошибку на первом разрыве графа ``fullgraph=True``:


In [39]:
print(a)

Graph Count: 2
Graph Break Count: 1
Op Count: 6
Break Reasons:
  Break Reason 1:
    Reason: generic_jump TensorVariable()
    User Stack:
      <FrameSummary file /tmp/ipykernel_703890/3263660924.py, line 3 in bar>
Ops per Graph:
  Ops 1:
    <built-in method abs of type object at 0x7f3b69d8c4a0>
    <built-in function add>
    <built-in function truediv>
    <built-in function lt>
  Ops 2:
    <built-in function mul>
    <built-in function mul>
Out Guards:
  Guard 1:
    Name: ''
    Source: global
    Create Function: DEFAULT_DEVICE
    Guard Types: ['DEFAULT_DEVICE']
    Code List: ['utils_device.CURRENT_DEVICE == None']
    Object Weakref: None
    Guarded Class Weakref: None
  Guard 2:
    Name: ''
    Source: global
    Create Function: BACKEND_MATCH
    Guard Types: ['BACKEND_MATCH']
    Code List: ['___check_current_backend(139886759614576)']
    Object Weakref: None
    Guarded Class Weakref: None
  Guard 3:
    Name: ''
    Source: shape_env
    Create Function: SHAPE_ENV
  

И действительно, мы видим, что запуск нашей модели с помощью ``torch.compile``
приводит к значительному ускорению. Ускорение в основном достигается за счет сокращения накладных расходов Python и
Чтение/запись графического процессора, поэтому наблюдаемое ускорение может варьироваться в зависимости от таких факторов, как 
архитектура модели и ее размер. <br>

<br>
Например, если архитектура модели простя, 
но занимает много памяти, то узким местом будет 
вычисления графического процессора и наблюдаемое ускорение может быть менее значительными. <br>


``torch.compile`` поддерживает разные режими компиляции. Подробнее про разные режимы можно посмотреть тут: [here](https://pytorch.org/get-started/pytorch-2.0/#user-experience).


```python
# default: optimizes for large models, low compile-time
#          and no extra memory usage
torch.compile(model)

# reduce-overhead: optimizes to reduce the framework overhead
#                and uses some extra memory. Helps speed up small models
torch.compile(model, mode="reduce-overhead")

# max-autotune: optimizes to produce the fastest model,
#               but takes a very long time to compile
torch.compile(model, mode="max-autotune")
```



В целом, для тестирования моделей PyTorch лучше использовать ``torch.utils.benchmark`` вместо самопальной timed. <br>

<br>

В этом примере нам нужна была такая функция что бы показать задержки на компиляцию.

<br>
<br>

**Посмотрим на сравнение скорости обучения.**


In [44]:
opt_bar = torch.compile(bar, fullgraph=True)
try:
    opt_bar(torch.randn(10), torch.randn(10))
except:
    tb.print_exc()

Наконец, если мы просто хотим, чтобы TorchDynamo выдал на torch FX граф,
мы можем использовать torch._dynamo.export. Обратите внимание, что ``torch._dynamo.export`` с
``fullgraph=True``, выдает ошибку, если TorchDynamo находит место разрыва графа.

In [ ]:
opt_bar = torch.compile(bar)
try:
    opt_bar(torch.randn(10), torch.randn(10))
except:
    tb.print_exc()

TorchDynamo не ломает граф модели, который мы использовали для анализа ускорения.

In [42]:
opt_model = torch.compile(init_model(), fullgraph=True)
print(opt_model(generate_data(16)[0]))

tensor([[ 0.0445, -0.2539,  0.3169,  ...,  0.0866,  0.0787,  0.0338],
        [ 0.0447, -0.1988,  0.4033,  ...,  0.0417,  0.1911, -0.1415],
        [-0.0104, -0.4276,  0.3993,  ...,  0.1492,  0.1909, -0.1091],
        ...,
        [ 0.0770, -0.1170,  0.3440,  ...,  0.0820,  0.2353, -0.0102],
        [ 0.1420, -0.2071,  0.4743,  ...,  0.1570,  0.2152,  0.1271],
        [ 0.0273, -0.2894,  0.3814,  ..., -0.0078,  0.3435, -0.0391]],
       device='cuda:1', grad_fn=<CompiledFunctionBackward>)


Наконец, если мы просто хотим, чтобы TorchDynamo выдал на torch FX граф,
мы можем использовать torch._dynamo.export. Обратите внимание, что ``torch._dynamo.export`` с
``fullgraph=True``, выдает ошибку, если TorchDynamo находит место разрыва графа.